# AML Compliance — Network Typology Walkthrough

A hand-authored, end-to-end walkthrough of factgraph as an **auditable reasoning substrate** for
anti-money-laundering monitoring. Everything here — schema, facts, rules — is written by hand. **No LLM is involved.**

This version shows **layered rule composition over a graph**, not just per-transaction thresholds. A single
large-transaction rule (a CTR) is the *simple* case and we keep exactly one. But real laundering is
**coordinated and multi-stage** — structuring into mule accounts, rapid forwarding, cross-border layering —
which per-transaction rules miss. Graph-linked, composable rules are the answer.

Three real typologies drive the rules below:

| Typology | What it looks like | Reference |
|---|---|---|
| **Structuring / smurfing** | many sub-$10,000 deposits to dodge the CTR threshold | FFIEC Appendix G |
| **Funnel / rapid movement of funds** | money in, rapidly forwarded out | FFIEC Appendix F |
| **Money-mule ring** | new accounts, a *shared sign-in device*, all wiring to a high-risk corridor | FATF typologies |

Design north star (FATF): **suspicion = multiple indicators co-occurring with no legitimate business
rationale** — which is exactly what `AND`/`OR` composition over shared entities expresses. References are at the end.

## 1 · Schema — a small graph: `Customer`, `Device`, `Transfer`

Field `repr` templates make proofs read in natural language (`%ENT` is the entity label from `Meta.repr`,
`%FLD` is the field value). `Customer.device` is an **entity reference** — a graph edge from a customer to the
device they sign in on. That edge is what later lets *one* rule link *two* customers (the money-mule signal a
per-transaction rule can never see).

In [1]:
from factgraph.sdk import (
    Entity, FactGraph, Field, Identity,
    build_application_rule, vars, Rule,
)
from factgraph.sdk.dsl import agg_count
from factgraph.core.evidence.write_protocol import set_field


class Device(Entity):
    device_id: str = Identity(repr="%ENT id %FLD")
    class Meta:
        repr = "Device %device_id"


class Customer(Entity):
    customer_id: str = Identity(repr="%ENT id %FLD")
    name: str = Field(repr="%ENT name is %FLD")
    tenure_days: int = Field(repr="%ENT account age is %FLD days")
    device: Device = Field(repr="%ENT signs in on %FLD")
    class Meta:
        repr = "Customer %customer_id"


class Transfer(Entity):
    txn_id: str = Identity(repr="%ENT id %FLD")
    customer: Customer = Field(repr="%ENT is by %FLD")
    amount: int = Field(repr="%ENT amount is %FLD")
    kind: str = Field(repr="%ENT direction is %FLD")
    dest_country: str = Field(repr="%ENT to %FLD")
    class Meta:
        repr = "Txn %txn_id"


fg = FactGraph.create(schema_classes=[Device, Customer, Transfer])
print("schema compiled: Device, Customer, Transfer")

schema compiled: Device, Customer, Transfer


## 2 · Facts — a mule ring, a lone structurer, and a clean control

- **Mule-1 / Mule-2 / Mule-3** — newly opened (20–50 days), **all sign in on Device `D-1`**, each takes three
  ~$9,500 deposits (structuring band) then wires ~$28,000 to **`IR`** (a high-risk corridor).
- **Eve** — an *old* account on her *own* device, but she also structures and wires cross-border: a **lone actor**,
  not part of the ring.
- **Dana** — a clean control (small, domestic, long-tenured).

`emit_exists` is the `<Entity>:exists` workaround (`fg.entities.create` does not yet emit it).

Two signals are genuinely *soft* and carry a probabilistic `bound` — **device-match** (0.9) and **jurisdiction-risk** (0.8). The native engine **ignores** them (§3–6 are unchanged); ProbLog uses them in §7.

In [2]:
def emit_exists(e_ref, entity_type):
    """Emit the <Entity>:exists claim that fg.entities.create does not yet emit."""
    set_field(fg.ledger, f"{entity_type}:exists", e_ref, [], None)


readable = {}  # idref -> customer_id, for legible output

d1 = fg.entities.create(Device, device_id="D-1"); emit_exists(d1, "Device")
d7 = fg.entities.create(Device, device_id="D-7"); emit_exists(d7, "Device")
d9 = fg.entities.create(Device, device_id="D-9"); emit_exists(d9, "Device")

# (id, name, tenure_days, device) — three mules share device D-1
CUSTOMERS = [
    ("C-M1",  "Mule-1",   20, d1),
    ("C-M2",  "Mule-2",   35, d1),
    ("C-M3",  "Mule-3",   50, d1),
    ("C-EVE", "Eve",    2000, d7),   # lone structurer: old account, own device
    ("C-DANA","Dana",   1200, d9),   # clean control
]
cref = {}
for cid, nm, tenure, dev in CUSTOMERS:
    c = fg.entities.create(Customer, customer_id=cid)
    fg.fields.set(Customer.name, c, nm)
    fg.fields.set(Customer.tenure_days, c, tenure)
    fg.fields.set(Customer.device, c, dev,
                  meta={"raw_kind": "probabilistic", "bound": [0.9, 0.9]})  # device-match confidence
    emit_exists(c, "Customer")
    cref[cid] = c
    readable[c] = cid

# (id, customer, amount, kind, dest_country)
TRANSFERS = [
    ("M1a","C-M1", 9500,"in","US"),("M1b","C-M1", 9600,"in","US"),("M1c","C-M1", 9400,"in","US"),("M1o","C-M1",28000,"out","IR"),
    ("M2a","C-M2", 9700,"in","US"),("M2b","C-M2", 9300,"in","US"),("M2c","C-M2", 9800,"in","US"),("M2o","C-M2",28000,"out","IR"),
    ("M3a","C-M3", 9200,"in","US"),("M3b","C-M3", 9900,"in","US"),("M3c","C-M3", 9500,"in","US"),("M3o","C-M3",27000,"out","IR"),
    ("E1", "C-EVE",9500,"in","US"),("E2", "C-EVE",9600,"in","US"),("E3", "C-EVE",9400,"in","US"),("Eo", "C-EVE",28000,"out","IR"),
    ("D1", "C-DANA",5000,"in","US"),("D2","C-DANA",6000,"in","US"),("Do","C-DANA",4000,"out","US"),
]
for tid, cid, amt, kind, cc in TRANSFERS:
    t = fg.entities.create(Transfer, txn_id=tid)
    fg.fields.set(Transfer.customer, t, cref[cid])
    fg.fields.set(Transfer.amount, t, amt)
    fg.fields.set(Transfer.kind, t, kind)
    if cc == "IR":  # jurisdiction-risk is a judgement -> attach a confidence
        fg.fields.set(Transfer.dest_country, t, cc,
                      meta={"raw_kind": "probabilistic", "bound": [0.8, 0.8]})
    else:
        fg.fields.set(Transfer.dest_country, t, cc)
    emit_exists(t, "Transfer")
print(f"seeded {len(CUSTOMERS)} customers, {len(TRANSFERS)} transfers")


def flagged(result):
    return sorted({readable.get(r.bindings["customer"]["value"], r.bindings["customer"]["value"]) for r in result})

seeded 5 customers, 19 transfers


## 3 · The one simple rule we keep — a single large transfer

The baseline AML rule: any single transfer at or over the reporting threshold. One condition — but it still
**explains itself**, condition by condition, via `explanation.narrate()`. This is the floor; everything below
composes upward from atoms like this one.

In [3]:
with vars("t", "amt") as (t, amt):
    large_transfer = build_application_rule(
        id="large_transfer",
        when=[Transfer(t).amount == amt, amt >= 10000],
        ports={"txn": t},
        repr="%txn is a single large transfer (at/over the reporting threshold)",
    )
res = fg.eval.evaluate(large_transfer, head=large_transfer, engine="native")
print("large transfers:", res.count())
print("--- proof of the first one ---")
for line in res.first().explain().narrate():
    print(line)

large transfers: 4
--- proof of the first one ---
Conclusion ── Txn M1o is a single large transfer (at/over the reporting threshold)
              [large_transfer · run_v1:10a22c4d… · c0]  holds
      produces:  txn = Txn M1o
  large_transfer  [holds]
       ✓ Txn M1o exists  [c0:atom:0]  holds
       ✓ Txn M1o amount is 28000  [c0:atom:1]  holds
       ✓ 28000 >= 10000  [c0:atom:2]  holds


## 4 · Layer 1 — typology detectors (the reusable bricks)

Five named detectors, each a concept other rules can consume:

- `structuring` — an **aggregate**: 3 or more sub-threshold deposits (shown standalone so the *count* appears in the proof).
- `new_account`, `forwards_out`, `cross_border_out` — behavioral atoms.
- `shared_device` — a **graph traversal**: two *distinct* customers on the same `Device`. This is the network
  signal per-transaction rules cannot see.

> Honest scope: aggregate **sub-evidence inside a composed proof** is still on the roadmap, so we keep the
> aggregate (`structuring`) standalone and compose the non-aggregate detectors below. The flags are the same;
> the composed proofs stay fully expanded.

In [4]:
# structuring — aggregate; shown standalone (clean count proof)
with vars("c", "t", "amt", "n") as (c, t, amt, n):
    structuring = build_application_rule(
        id="structuring",
        when=[
            Customer(c),
            n == agg_count(where=[
                Transfer(t).customer == c,
                Transfer(t).kind == "in",
                Transfer(t).amount == amt,
                amt >= 9000, amt < 10000,
            ]),
            n >= 3,
        ],
        ports={"customer": c, "count": n},
        repr="%customer shows a structuring pattern (%count sub-threshold deposits)",
    )

# graph / behavioral atoms — non-aggregate, composable
with vars("c", "td") as (c, td):
    new_account = build_application_rule(
        id="new_account",
        when=[Customer(c), Customer(c).tenure_days == td, td < 90],
        ports={"customer": c},
        repr="%customer is a newly opened account",
    )

with vars("c", "t", "amt") as (c, t, amt):
    forwards_out = build_application_rule(
        id="forwards_out",
        when=[
            Customer(c),
            Transfer(t).customer == c,
            Transfer(t).kind == "out",
            Transfer(t).amount == amt,
            amt >= 20000,
        ],
        ports={"customer": c},
        repr="%customer forwards a large outflow",
    )

with vars("c", "t", "cc") as (c, t, cc):
    cross_border = build_application_rule(
        id="cross_border_out",
        when=[
            Customer(c),
            Transfer(t).customer == c,
            Transfer(t).kind == "out",
            Transfer(t).dest_country == cc,
            cc == "IR",
        ],
        ports={"customer": c},
        repr="%customer wires funds to a high-risk jurisdiction",
    )

# shared_device — graph traversal: two distinct customers, same sign-in device
with vars("c", "c2", "d", "id1", "id2") as (c, c2, d, id1, id2):
    shared_device = build_application_rule(
        id="shared_device",
        when=[
            Customer(c),  Customer(c).device == d,  Customer(c).customer_id == id1,
            Customer(c2), Customer(c2).device == d, Customer(c2).customer_id == id2,
            id1 != id2,
        ],
        ports={"customer": c},
        repr="%customer shares a sign-in device with another customer",
    )

for name, rule in [("structuring", structuring), ("new_account", new_account),
                   ("forwards_out", forwards_out), ("cross_border_out", cross_border),
                   ("shared_device", shared_device)]:
    print(f"  {name:18}: {flagged(fg.eval.evaluate(rule, head=rule, engine='native'))}")

print("\n--- structuring proof (Mule-1, count shown) ---")
sres = fg.eval.evaluate(structuring, head=structuring, engine="native")
srow = next(r for r in sres if r.bindings["customer"]["value"] == cref["C-M1"])
for line in srow.explain().narrate():
    print(line)

  structuring       : ['C-EVE', 'C-M1', 'C-M2', 'C-M3']
  new_account       : ['C-M1', 'C-M2', 'C-M3']
  forwards_out      : ['C-EVE', 'C-M1', 'C-M2', 'C-M3']
  cross_border_out  : ['C-EVE', 'C-M1', 'C-M2', 'C-M3']
  shared_device     : ['C-M1', 'C-M2', 'C-M3']

--- structuring proof (Mule-1, count shown) ---
Conclusion ── Customer C-M1 shows a structuring pattern (3 sub-threshold deposits)
              [structuring · run_v1:bd3df515… · c0]  holds
      produces:  customer = Customer C-M1,  count = 3
  structuring  [holds]
       ✓ Customer C-M1 exists  [c0:atom:0]  holds
       ✓ 3 equals count  [c0:atom:1]  holds
       ✓ 3 >= 3  [c0:atom:2]  holds


## 5 · Layer 2 — composing a network SAR alert

A SAR candidate is a **disjunction of two conjunctive typologies**, joined on the same customer:

```
SAR := ( forwards_out  AND  cross_border_out )                  ← a lone cross-border forwarder
     | ( new_account   AND  forwards_out  AND  shared_device )  ← a coordinated mule ring
```

`&` is `AND` (joined on the `customer` port), `|` is `OR`. The proof shows **both paths** for every candidate:
the mules conclude via *either* branch; **Eve concludes via the cross-border branch only** — her ring branch
fails at `2000 < 90` (old account), and the proof says exactly that. Dana matches nothing.

In [5]:
# SAR := ( forwards_out AND cross_border )                       <- lone cross-border forwarder
#      | ( new_account AND forwards_out AND shared_device )       <- coordinated new-account mule ring
branch_xborder = (forwards_out.as_("f1") & cross_border.as_("xb")).join_by_ports("customer")
branch_ring    = (new_account.as_("na") & forwards_out.as_("f2") & shared_device.as_("sd")).join_by_ports("customer")
sar = branch_xborder | branch_ring

res = fg.eval.evaluate(sar, head=Rule.projection("customer"), engine="native")
print("SAR candidates:", flagged(res))
assert flagged(res) == ["C-EVE", "C-M1", "C-M2", "C-M3"]

for cid in ("C-M1", "C-EVE"):
    row = next(r for r in res if r.bindings["customer"]["value"] == cref[cid])
    print(f"\n--- SAR proof: {cid} ---")
    for line in row.explain().narrate():
        print(line)

SAR candidates: ['C-EVE', 'C-M1', 'C-M2', 'C-M3']

--- SAR proof: C-M1 ---
Conclusion ── projection(customer)
              [projection(customer) · run_v1:4e06696f… · c0|c1 (concluded via c0)]  holds
      produces:  customer = Customer C-M1
  Derivation:  projection(customer) <= ( cross_border_out AND forwards_out ) OR ( forwards_out AND new_account AND shared_device )
▸ Path c0  [holds]
               join:  forwards_out.customer = cross_border_out.customer  [holds]
  projection(customer) [head] ── "projection(customer)"  [holds]
  forwards_out ── "Customer C-M1 forwards a large outflow"  [holds]
       ✓ Customer C-M1 exists  [c0:atom:6]  holds
       ✓ Txn M1o exists  [c0:atom:7]  holds
       ✓ Txn M1o is by Customer C-M1  [c0:atom:8]  holds
       ✓ Txn M1o direction is out  [c0:atom:9]  holds
       ✓ Txn M1o amount is 28000  [c0:atom:10]  holds
       ✓ 28000 >= 20000  [c0:atom:11]  holds
  cross_border_out ── "Customer C-M1 wires funds to a high-risk jurisdiction"  [holds]
   

## 6 · What this shows

- **Composition** — a complex alert is `(A AND B) OR (C AND D AND E)` over reusable detectors; each detector is
  independently auditable, and the operators map 1:1 to the proof's paths and joins.
- **Graph** — `shared_device` links two customers through a shared `Device` entity. That is the coordinated-ring
  signal static, per-transaction rules miss — and the reason the substrate is a graph.
- **Auditable by construction** — every SAR candidate carries a full, replayable proof down to the raw transfers:
  a SAR you could hand a regulator, with the deciding condition (and the failing alternative) made explicit.

### References

- **FFIEC BSA/AML Examination Manual** — [Appendix F · red flags](https://bsaaml.ffiec.gov/manual/Appendices/07) · [Appendix G · structuring](https://bsaaml.ffiec.gov/manual/Appendices/08)
- **FATF** — [typology reports](https://eurasiangroup.org/en/fatf-typology-reports)
- **Wolfsberg Group** — [Effective Monitoring for Suspicious Activity, Part I](https://wolfsberg-group.org/resources/168/)
- **Graph analytics for AML** — [TigerGraph: structuring & layering](https://www.tigergraph.com/blog/money-laundering-detection-with-aml-graph-analytics-structuring-and-layering/) · [GARG-AML against smurfing](https://arxiv.org/pdf/2506.04292)

## 7 · The same rules, probabilistically — one config, two versions

Not every signal is certain. A **device-fingerprint match** and a **jurisdiction-risk rating** are
judgements with a confidence, not hard facts — and in the schema above we annotated exactly those two
with a probabilistic `bound`, **changing nothing else**. The native engine ignores the bounds, so §3–6
are unchanged.

Now we evaluate **the identical `sar` expression** with a `ProbLogConfig` instead of `engine="native"`.
The binary verdict becomes a calibrated **risk score**. And because the `OR` combines *independent*
evidence by **noisy-OR**, a mule — corroborated by **both** the cross-border path **and** the device-ring
path — outranks the lone structurer, who has only one path:

- **Eve** — one uncertain path: cross-border `0.80` → **P(SAR) = 0.80**
- **Mules** — two independent paths: cross-border `0.80`, device-ring `0.891`, combined
  `1 − (1−0.80)(1−0.891)` → **P(SAR) = 0.978**

The cell prints the score table and this per-path decomposition **straight from each branch's `certainty`** —
deterministic, no proof-rendering needed. (The engine also has a per-condition probabilistic *proof*, but
its rendering depends on the evaluation path — a lowered composite gets a terser companion form — so here we
read the probabilities directly. The full condition structure is the deterministic proof in §5, which
ProbLog reuses and reweights.)

In [6]:
from factgraph.sdk import ProbLogConfig

# probabilistic bounds -> point probabilities; deterministic facts -> certain
cfg = ProbLogConfig(
    uncertainty_projection={
        "probabilistic": {"policy": "identity_probability"},
        "possibilistic": {"policy": "reject"},
        "fallback": "use_default",
    },
    name="aml_risk",
)

# The SAME `sar` expression -- only the engine changes (config= infers ProbLog).
native  = {readable[r.bindings["customer"]["value"]]: r.certainty
           for r in fg.eval.evaluate(sar, head=Rule.projection("customer"), engine="native")}
problog = {readable[r.bindings["customer"]["value"]]: r.certainty
           for r in fg.eval.evaluate(sar, head=Rule.projection("customer"), config=cfg)}

print(f"{'customer':8} | {'native':>16} | {'problog (risk)':>14}")
print("-" * 46)
for cid in sorted(problog):
    print(f"{cid:8} | flagged ({native[cid].lo:>4.2f} {native[cid].kind[:4]}) | {problog[cid].lo:>14.4f}")

# Why the ranking? Read each branch's probability and combine by noisy-OR -- straight from `certainty`.
def path_p(branch, cid):
    rows = fg.eval.evaluate(branch, head=Rule.projection("customer"), config=cfg)
    row = next((r for r in rows if readable[r.bindings["customer"]["value"]] == cid), None)
    return None if row is None else row.certainty.lo

print("\nwhy the ranking (P decomposes by path; OR combines independent paths, noisy-OR):")
for cid in ("C-EVE", "C-M1"):
    px = path_p(branch_xborder, cid)
    pr = path_p(branch_ring, cid)
    combined = px if pr is None else 1 - (1 - px) * (1 - pr)
    ring_txt = f"device-ring={pr:.3f}" if pr is not None else "device-ring=\u2014"
    print(f"  {cid:6}: cross-border={px:.3f}, {ring_txt}  ->  P(SAR) = {combined:.4f}")

customer |           native | problog (risk)
----------------------------------------------
C-EVE    | flagged (1.00 bool) |         0.8000
C-M1     | flagged (1.00 bool) |         0.9782
C-M2     | flagged (1.00 bool) |         0.9782
C-M3     | flagged (1.00 bool) |         0.9782

why the ranking (P decomposes by path; OR combines independent paths, noisy-OR):


  C-EVE : cross-border=0.800, device-ring=—  ->  P(SAR) = 0.8000


  C-M1  : cross-border=0.800, device-ring=0.891  ->  P(SAR) = 0.9782
